In [1]:
# Import required libraries for data processing, ML models, and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_selection import SelectKBest, chi2

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier

from imblearn.over_sampling import SMOTE

print("Imports OK")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Load heart failure clinical records dataset from CSV file
df = pd.read_csv("../1_Dataset/raw/heart_failure_clinical_records_dataset.csv")
print(df.shape)

In [ ]:
# Display first 5 rows of the dataset to understand data structure
df.head()

In [ ]:
# Check column names in the dataset
df.columns

In [ ]:
# Analyze class distribution of target variable (DEATH_EVENT imbalance)
df["DEATH_EVENT"].value_counts()

In [ ]:
# Check for missing values in the dataset
df.isnull().sum()

In [ ]:
# Separate features (X) and target variable (y) - prepare for model training
X = df.drop("DEATH_EVENT", axis=1)
y = df["DEATH_EVENT"]

print(X.shape, y.shape)

In [ ]:
# Split data into training (80%) and testing (20%) sets with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

In [ ]:
# Normalize features using StandardScaler to bring all features to same scale
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape, X_test_scaled.shape)

In [ ]:
# Apply SMOTE to handle class imbalance by over-sampling minority class
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(np.bincount(y_train_smote))

In [ ]:
# Verify shape of balanced training data after SMOTE
X_train_smote.shape, y_train_smote.shape

In [ ]:
# Select top 7 most important features using SelectKBest with chi2 scoring
selector = SelectKBest(score_func=chi2, k=7)

X_train_fs = selector.fit_transform(np.abs(X_train_smote), y_train_smote)
X_test_fs = selector.transform(np.abs(X_test_scaled))

print(X_train_fs.shape, X_test_fs.shape)

In [ ]:
# Display the names of selected important features for interpretability
selected_features = X.columns[selector.get_support()]
selected_features

In [ ]:
# Train baseline Gradient Boosting model with default hyperparameters
gbm = GradientBoostingClassifier(random_state=42)
gbm.fit(X_train_fs, y_train_smote)

In [ ]:
# Evaluate baseline model performance using accuracy, precision, recall, and F1-score
y_pred = gbm.predict(X_test_fs)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

acc, prec, rec, f1

In [ ]:
# Store baseline model evaluation results for comparison
baseline_results = {
    "Model": "GBM (baseline)",
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1": f1
}

baseline_results

In [ ]:
# Define objective function for PSO optimization of GBM hyperparameters
from mealpy.swarm_based.PSO import AIW_PSO
from mealpy.utils.space import IntegerVar, FloatVar
from sklearn.model_selection import cross_val_score

def objective(solution):
    n_estimators = int(solution[0])
    learning_rate = solution[1]
    max_depth = int(solution[2])

    model = GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        random_state=42
    )
    score = cross_val_score(model, X_train_fs, y_train_smote, cv=3, scoring="accuracy").mean()
    return 1 - score

In [ ]:
# Run PSO (Particle Swarm Optimization) to find optimal GBM hyperparameters
problem = {
    "obj_func": objective,
    "bounds": [IntegerVar(50, 200, 'n_estimators'), FloatVar(0.01, 0.3, 'learning_rate'), IntegerVar(2, 5, 'max_depth')],
    "minmax": "min"
}

model_pso = AIW_PSO(epoch=10, pop_size=10)
agent = model_pso.solve(problem)
best_position = agent.solution
best_fitness = agent.target

best_position, best_fitness

In [ ]:
# Train optimized Gradient Boosting model with best hyperparameters from PSO
best_n_estimators = int(best_position[0])
best_learning_rate = best_position[1]
best_max_depth = int(best_position[2])

gbm_optimized = GradientBoostingClassifier(
    n_estimators=best_n_estimators,
    learning_rate=best_learning_rate,
    max_depth=best_max_depth,
    random_state=42
)

gbm_optimized.fit(X_train_fs, y_train_smote)

In [ ]:
# Evaluate optimized model performance and compare with baseline model
y_pred_opt = gbm_optimized.predict(X_test_fs)

acc_opt = accuracy_score(y_test, y_pred_opt)
prec_opt = precision_score(y_test, y_pred_opt)
rec_opt = recall_score(y_test, y_pred_opt)
f1_opt = f1_score(y_test, y_pred_opt)

acc_opt, prec_opt, rec_opt, f1_opt

In [ ]:
# Compare performance metrics between baseline and optimized models
comparison = {
    "Baseline_GBM_Accuracy": acc,
    "Optimized_GBM_Accuracy": acc_opt,
    "Baseline_F1": f1,
    "Optimized_F1": f1_opt
}
comparison

“Gradient Boosting with AIW-PSO hyperparameter optimization demonstrated competitive performance for heart failure survival prediction. Although optimization did not significantly outperform the baseline in this experiment, the proposed pipeline effectively handled class imbalance, feature selection, and model tuning, aligning with established research methodologies.”

In [ ]:
# Save trained models and preprocessing objects for deployment in Streamlit app
import joblib
joblib.dump(scaler, "../scaler.pkl")
joblib.dump(selector, "../selector.pkl")
joblib.dump(gbm_optimized, "../gbm_model.pkl")